# 04 - Model Training

Train LightGBM models for bout prediction.

**Models:**
1. **Winner Prediction** - Binary classification: P(east wrestler wins)
2. **Kimarite Prediction** - Multiclass classification over top 20 techniques
3. **Bout Duration** - Regression (if duration data available)

**Training Strategy:**
- Time-based cross-validation (train on older, validate on newer)
- Early stopping to prevent overfitting

**Inputs:**
- `features.parquet` from notebook 03

**Outputs:**
- `winner_model.lgb` - Winner prediction model
- `kimarite_model.lgb` - Kimarite prediction model
- `kimarite_encoder.joblib` - Label encoder for kimarite

In [ ]:
# Environment setup
import sys
import os

INPUT_PATH = '/kaggle/input/sumo-data-03' if os.path.exists('/kaggle/input') else './output'
OUTPUT_PATH = '/kaggle/working' if os.path.exists('/kaggle/working') else './output'

if not os.path.exists('/kaggle/input'):
    sys.path.insert(0, '../src')

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
!pip install -q lightgbm scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss,
    classification_report, confusion_matrix
)
import joblib
import warnings
warnings.filterwarnings('ignore')

## Load Features

In [ ]:
features_df = pd.read_parquet(f"{INPUT_PATH}/features.parquet")
print(f"Loaded {len(features_df):,} bouts with features")
print(f"Columns: {len(features_df.columns)}")

# Add year column if missing
if 'year' not in features_df.columns and 'basho_year' in features_df.columns:
    features_df['year'] = features_df['basho_year']

# Show column names
print(f"\nSample columns:")
print([c for c in features_df.columns if 'career' in c or 'elo' in c][:20])

In [ ]:
# Check date range
print(f"Basho range: {features_df['bashoId'].min()} to {features_df['bashoId'].max()}")
print(f"\nBouts per year:")
print(features_df.groupby('year').size().tail(10))

## Define Features

In [ ]:
# Dynamically select numeric features from the dataset
# Exclude identifiers and target columns
EXCLUDE_COLS = [
    'bout_id', 'bashoId', 'eastId', 'westId', 'winnerId', 
    'eastShikona', 'westShikona', 'eastRank', 'westRank',
    'kimarite', 'kimarite_category', 'division',
    'winnerEn', 'winnerJp', 'east_won', 'west_won',
    'venue', 'east_country_of_origin', 'west_country_of_origin',
    'symmetric_version'
]

# Get all numeric columns
numeric_cols = features_df.select_dtypes(include=[np.number]).columns.tolist()
FEATURE_COLS = [c for c in numeric_cols if c not in EXCLUDE_COLS]

print(f"Using {len(FEATURE_COLS)} numeric features")
print(f"\nSample features:")
print(FEATURE_COLS[:30])

In [ ]:
# Top kimarite for multiclass prediction
TOP_KIMARITE = [
    'yorikiri', 'oshidashi', 'hatakikomi', 'uwatenage', 'oshitaoshi',
    'shitatenage', 'tsukiotoshi', 'hikiotoshi', 'kotenage', 'sukuinage',
    'tsukidashi', 'okuridashi', 'yoritaoshi', 'katasukashi', 'sotogake',
    'uwatedashinage', 'makiotoshi', 'tsukitaoshi', 'kimedashi', 'uchigake',
]

print(f"\nTop {len(TOP_KIMARITE)} kimarite for prediction")

## Time-Based Train/Val/Test Split

In [ ]:
# Determine split points based on data range
basho_range = features_df['bashoId'].astype(str).unique()
basho_range = sorted(basho_range)

print(f"Total basho: {len(basho_range)}")
print(f"First: {basho_range[0]}, Last: {basho_range[-1]}")

# Use last 2 years for val+test
# Adjust these based on your data
if len(basho_range) > 12:
    train_end = basho_range[-12]  # 2 years before end
    val_end = basho_range[-6]     # 1 year before end
else:
    # Small dataset - simpler split
    train_end = basho_range[int(len(basho_range) * 0.7)]
    val_end = basho_range[int(len(basho_range) * 0.85)]

print(f"\nSplit points:")
print(f"  Train: before {train_end}")
print(f"  Val: {train_end} to {val_end}")
print(f"  Test: after {val_end}")

In [ ]:
# Split data
train_df = features_df[features_df['bashoId'].astype(str) < train_end].copy()
val_df = features_df[
    (features_df['bashoId'].astype(str) >= train_end) & 
    (features_df['bashoId'].astype(str) < val_end)
].copy()
test_df = features_df[features_df['bashoId'].astype(str) >= val_end].copy()

print(f"Train: {len(train_df):,} bouts")
print(f"Val: {len(val_df):,} bouts")
print(f"Test: {len(test_df):,} bouts")

## Prepare Data for Training

In [ ]:
def prepare_features(df, feature_cols):
    """Prepare features, handling missing values."""
    X = df[feature_cols].copy()
    
    # Fill missing with median
    for col in X.columns:
        if X[col].dtype in ['float64', 'float32', 'int64', 'int32']:
            median_val = X[col].median()
            if pd.isna(median_val):
                median_val = 0
            X[col] = X[col].fillna(median_val)
    
    return X

In [ ]:
# Filter to valid targets
train_valid = train_df[train_df['east_won'].notna()].copy()
val_valid = val_df[val_df['east_won'].notna()].copy()
test_valid = test_df[test_df['east_won'].notna()].copy()

print(f"Valid bouts - Train: {len(train_valid):,}, Val: {len(val_valid):,}, Test: {len(test_valid):,}")

In [ ]:
# Prepare features
X_train = prepare_features(train_valid, FEATURE_COLS)
X_val = prepare_features(val_valid, FEATURE_COLS)
X_test = prepare_features(test_valid, FEATURE_COLS)

y_train = train_valid['east_won'].astype(int)
y_val = val_valid['east_won'].astype(int)
y_test = test_valid['east_won'].astype(int)

print(f"X_train shape: {X_train.shape}")
print(f"y_train distribution: {y_train.value_counts().to_dict()}")

## Model 1: Winner Prediction

In [ ]:
# LightGBM parameters
winner_params = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
}

In [ ]:
# Create datasets
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Train model
print("Training winner prediction model...")
winner_model = lgb.train(
    winner_params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

print(f"\nBest iteration: {winner_model.best_iteration}")

In [ ]:
# Evaluate winner model
y_pred_train = winner_model.predict(X_train)
y_pred_val = winner_model.predict(X_val)
y_pred_test = winner_model.predict(X_test)

print("=== Winner Model Metrics ===")
for name, y_true, y_pred in [
    ('Train', y_train, y_pred_train),
    ('Val', y_val, y_pred_val),
    ('Test', y_test, y_pred_test)
]:
    accuracy = accuracy_score(y_true, (y_pred > 0.5).astype(int))
    auc = roc_auc_score(y_true, y_pred)
    logloss = log_loss(y_true, y_pred)
    print(f"{name}: Accuracy={accuracy:.4f}, AUC={auc:.4f}, LogLoss={logloss:.4f}")

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': winner_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\nTop 20 features by importance:")
display(importance_df.head(20))

## Model 2: Kimarite Prediction

In [ ]:
# Prepare kimarite data
train_kim = train_valid[train_valid['kimarite'].notna()].copy()
val_kim = val_valid[val_valid['kimarite'].notna()].copy()
test_kim = test_valid[test_valid['kimarite'].notna()].copy()

X_train_kim = prepare_features(train_kim, FEATURE_COLS)
X_val_kim = prepare_features(val_kim, FEATURE_COLS)
X_test_kim = prepare_features(test_kim, FEATURE_COLS)

# Map to top kimarite or 'other'
def map_kimarite(k):
    k_lower = str(k).lower().strip()
    return k_lower if k_lower in TOP_KIMARITE else 'other'

y_train_kim = train_kim['kimarite'].apply(map_kimarite)
y_val_kim = val_kim['kimarite'].apply(map_kimarite)
y_test_kim = test_kim['kimarite'].apply(map_kimarite)

print(f"Kimarite samples - Train: {len(X_train_kim)}, Val: {len(X_val_kim)}, Test: {len(X_test_kim)}")
print(f"\nKimarite distribution in training:")
print(y_train_kim.value_counts())

In [ ]:
# Encode labels
le = LabelEncoder()
y_train_kim_enc = le.fit_transform(y_train_kim)
y_val_kim_enc = le.transform(y_val_kim)
y_test_kim_enc = le.transform(y_test_kim)

print(f"Classes: {le.classes_}")

In [ ]:
# Kimarite model parameters
kimarite_params = {
    'objective': 'multiclass',
    'num_class': len(le.classes_),
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
}

train_kim_data = lgb.Dataset(X_train_kim, label=y_train_kim_enc)
val_kim_data = lgb.Dataset(X_val_kim, label=y_val_kim_enc, reference=train_kim_data)

print("Training kimarite prediction model...")
kimarite_model = lgb.train(
    kimarite_params,
    train_kim_data,
    num_boost_round=1000,
    valid_sets=[train_kim_data, val_kim_data],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

print(f"\nBest iteration: {kimarite_model.best_iteration}")

In [ ]:
# Evaluate kimarite model
y_pred_kim_test = kimarite_model.predict(X_test_kim)
y_pred_kim_class = y_pred_kim_test.argmax(axis=1)

accuracy_kim = accuracy_score(y_test_kim_enc, y_pred_kim_class)
print(f"\nKimarite Test Accuracy: {accuracy_kim:.4f}")

# Top-3 accuracy
top3_correct = sum(
    true in np.argsort(pred)[-3:] 
    for true, pred in zip(y_test_kim_enc, y_pred_kim_test)
)
top3_acc = top3_correct / len(y_test_kim_enc)
print(f"Kimarite Top-3 Accuracy: {top3_acc:.4f}")

In [ ]:
# Kimarite category derivation
KIMARITE_PUSH = ['oshidashi', 'tsukidashi', 'oshitaoshi', 'tsukiotoshi', 'tsukitaoshi', 'okuridashi']
KIMARITE_GRAPPLE = ['yorikiri', 'uwatenage', 'shitatenage', 'sukuinage', 'kotenage', 'yoritaoshi', 
                    'uwatedashinage', 'makiotoshi', 'kimedashi', 'sotogake', 'uchigake']
KIMARITE_EVASION = ['hatakikomi', 'hikiotoshi', 'katasukashi']

def get_category_probs(probs, classes):
    """Sum kimarite probs into categories."""
    push_idx = [i for i, c in enumerate(classes) if c in KIMARITE_PUSH]
    grapple_idx = [i for i, c in enumerate(classes) if c in KIMARITE_GRAPPLE]
    evasion_idx = [i for i, c in enumerate(classes) if c in KIMARITE_EVASION]
    
    push_prob = probs[:, push_idx].sum(axis=1) if push_idx else np.zeros(len(probs))
    grapple_prob = probs[:, grapple_idx].sum(axis=1) if grapple_idx else np.zeros(len(probs))
    evasion_prob = probs[:, evasion_idx].sum(axis=1) if evasion_idx else np.zeros(len(probs))
    
    return push_prob, grapple_prob, evasion_prob

push_prob, grapple_prob, evasion_prob = get_category_probs(y_pred_kim_test, le.classes_)

print(f"\nAverage category probabilities:")
print(f"  Push: {push_prob.mean():.3f}")
print(f"  Grapple: {grapple_prob.mean():.3f}")
print(f"  Evasion: {evasion_prob.mean():.3f}")

## Save Models

In [ ]:
# Save winner model
winner_model.save_model(f"{OUTPUT_PATH}/winner_model.lgb")
print(f"Saved winner model to {OUTPUT_PATH}/winner_model.lgb")

# Save kimarite model
kimarite_model.save_model(f"{OUTPUT_PATH}/kimarite_model.lgb")
print(f"Saved kimarite model to {OUTPUT_PATH}/kimarite_model.lgb")

# Save label encoder
joblib.dump(le, f"{OUTPUT_PATH}/kimarite_encoder.joblib")
print(f"Saved label encoder to {OUTPUT_PATH}/kimarite_encoder.joblib")

# Save feature list
pd.Series(FEATURE_COLS).to_csv(f"{OUTPUT_PATH}/feature_columns.csv", index=False)
print(f"Saved feature list to {OUTPUT_PATH}/feature_columns.csv")

In [ ]:
# Save test predictions for evaluation notebook
test_predictions = test_valid[['bout_id', 'bashoId', 'day', 'eastId', 'westId', 'winnerId', 'kimarite', 'east_won']].copy()
test_predictions['pred_east_win_prob'] = y_pred_test
test_predictions['pred_east_win'] = (y_pred_test > 0.5).astype(int)

test_predictions.to_parquet(f"{OUTPUT_PATH}/test_predictions.parquet", index=False)
print(f"Saved test predictions to {OUTPUT_PATH}/test_predictions.parquet")

## Summary

In [ ]:
print("=== Model Training Summary ===")
print(f"\nWinner Model:")
print(f"  Test Accuracy: {accuracy_score(y_test, (y_pred_test > 0.5).astype(int)):.4f}")
print(f"  Test AUC: {roc_auc_score(y_test, y_pred_test):.4f}")
print(f"  Baseline (50%): 0.5000")

print(f"\nKimarite Model:")
print(f"  Test Top-1 Accuracy: {accuracy_kim:.4f}")
print(f"  Test Top-3 Accuracy: {top3_acc:.4f}")

print(f"\nModels saved to: {OUTPUT_PATH}")

## Next Steps

Models are ready for:
- `05_evaluation.ipynb` - Detailed evaluation, calibration, SHAP analysis